In [13]:
import pandas as pd
from datasets import Dataset
df = pd.read_csv('/home/wagyu0923/project/Document_Analyzer/evaluation_data.csv')


In [14]:
import sys
import os
sys.path.append('/home/wagyu0923/project/Document_Analyzer')
from pipeline.document_loader import DocumentLoader
from pipeline.chunker import Chunker
from pipeline.embedder import Embedder
from pipeline.vector_retriever import VectorRetriever
from pipeline.generator import Generator
import config
from tkinter import filedialog

def setup_pipeline():
    chunker = Chunker(
        chunk_size = config.CHUNK_SIZE,
        overlap_size = config.OVERLAP_SIZE
    )
    print('Chunking Complete')
    embedder = Embedder(
        model_name = config.EMBEDDING_MODEL
    )
    print('Embedding Complete')
    retriever =VectorRetriever(
        db_path = config.DB_PATH,
        model_name = config.EMBEDDING_MODEL,
        collection_name = config.COLLECTION_NAME
    )
    print('Retrieving Coplete')
    generator = Generator(
        model_name = config.LLM_NAME,
        options = config.DEFAULT_OLLAMA_OPTIONS
    )
    return chunker, embedder, retriever, generator

def run_indexing(file_path, chunker, embedder, retriever):
    loader = DocumentLoader(file_path = file_path)
    document = loader.load()
    chunks = chunker.chunking(document)
    embedded_chunks = embedder.embed_documents(chunks)
    file_name = os.path.basename(file_path)
    retriever.add_documents(embedded_chunks, file_name)

chunker, embedder, retriever, generator = setup_pipeline()

file_path = '/home/wagyu0923/project/Document_Analyzer/pdf_files/[세토피아][정정]반기보고서(2025.09.09).pdf'
run_indexing(file_path, chunker, embedder, retriever)

Chunking Complete
Embedding Complete
Retrieving Coplete


In [15]:
import json
dataset = df.copy()
for index, query in enumerate(df['user_input']):
    retrieved_data = retriever.retrieve(query)
    outputs = generator.generate(retrieved_data, query)
    try:
        outputs = json.loads(outputs)
    except json.JSONDecodeError:
        print(f'JSON Decode Error at index {index}. Skipping.') 
        continue 
    if 'used_context' not in outputs.keys():
        outputs['used_context'] = []
    elif 'answer' not in outputs.keys():
        outputs['answer'] = ''
    dataset.loc[index, 'retrieved_contexts'] = outputs['used_context']
    dataset.loc[index, 'response'] = outputs['answer']
    print(f'progress : {index+1}/{len(df)}')




/tmp/ipykernel_27319/3895652976.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '대표이사 (성 명) 서상철' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dataset.loc[index, 'retrieved_contexts'] = outputs['used_context']
/tmp/ipykernel_27319/3895652976.py:16: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The current CEO of Setopia Co., Ltd. is 서상철.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dataset.loc[index, 'response'] = outputs['answer']


progress : 1/30
progress : 2/30
progress : 3/30
progress : 4/30
progress : 5/30
progress : 6/30
progress : 7/30
progress : 8/30
progress : 9/30
progress : 10/30
progress : 11/30
progress : 12/30
progress : 13/30
progress : 14/30
progress : 15/30
progress : 16/30
progress : 17/30
progress : 18/30
progress : 19/30
progress : 20/30
progress : 21/30
progress : 22/30
progress : 23/30
progress : 24/30
progress : 25/30
progress : 26/30
progress : 27/30
progress : 28/30
progress : 29/30
progress : 30/30


In [16]:
import ast  # ast 모듈을 임포트합니다.
import json # (기존 코드)
import pandas as pd # (기존 코드)

# 'json.loads(x)'를 'ast.literal_eval(x)'로 변경
dataset["retrieved_contexts"] = dataset["retrieved_contexts"].apply(
    lambda x: []
    if x is None or (isinstance(x, float) and pd.isna(x)) or x == ""
    # 이 부분을 수정합니다.
    else (ast.literal_eval(x.strip()) if isinstance(x, str) and x.strip().startswith("[") and x.strip().endswith("]")
          else (x if isinstance(x, list) else [x]))
)

# 'reference' 컬럼 코드는 그대로 둡니다.
if "reference" in dataset.columns:
    dataset["reference"] = dataset["reference"].apply(
        lambda x: "" if x is None or (isinstance(x, float) and pd.isna(x))
        else (x if isinstance(x, str) else "\n\n".join(map(str, x)))
    )

In [18]:
dataset.to_csv('response_data.csv')

In [22]:
# ===== RAGAS 평가: Ollama + SentenceTransformer (LangChain 없음) =====
import asyncio
from dataclasses import dataclass
from typing import Any, List, Optional

import ollama
import pandas as pd
from datasets import Dataset
from sentence_transformers import SentenceTransformer

from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.llms import BaseRagasLLM
from ragas.embeddings import BaseRagasEmbeddings
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
)

# -------------------------------------------------------------------
# 0) 컨텍스트 길이 줄이기 (토큰 수 줄여서 Timeout 완화)
# -------------------------------------------------------------------
def truncate_contexts(ctx_list, max_items=3, max_chars=1500):
    if ctx_list is None:
        return []
    if not isinstance(ctx_list, list):
        ctx_list = [ctx_list]

    trimmed = []
    for c in ctx_list[:max_items]:
        s = str(c)
        if len(s) > max_chars:
            s = s[:max_chars]
        trimmed.append(s)
    return trimmed

eval_df = dataset.copy()
eval_df["retrieved_contexts"] = eval_df["retrieved_contexts"].apply(truncate_contexts)
eval_df = eval_df.fillna({"response": "", "reference": ""})

# ragas 쪽에서 많이 쓰는 기본 컬럼 이름으로 맞추기
ragas_df = pd.DataFrame({
    "question": eval_df["user_input"],
    "contexts": eval_df["retrieved_contexts"],
    "answer": eval_df["response"],
    "ground_truth": eval_df["reference"],
})

hf_ds = Dataset.from_pandas(ragas_df)

# -------------------------------------------------------------------
# 1) RAGAS가 기대하는 LLM 결과 포맷 (간단 구조체)
# -------------------------------------------------------------------
@dataclass
class SimpleGeneration:
    text: str

@dataclass
class SimpleLLMResult:
    # ragas 내부에서 result.generations[0][0].text 이런 식으로 접근
    generations: List[List[SimpleGeneration]]


# -------------------------------------------------------------------
# 2) Ollama용 RAGAS LLM 래퍼
# -------------------------------------------------------------------
class OllamaRagasLLM(BaseRagasLLM):
    def __init__(
        self,
        model: str = "phi3:medium",
        host: str = "http://127.0.0.1:11434",
    ):
        super().__init__()
        # 여기 model은 "모델 이름" 문자열
        self.model = model
        self.client = ollama.Client(host=host)

    def _to_text(self, prompt: Any) -> str:
        if hasattr(prompt, "to_string"):
            return prompt.to_string()
        if hasattr(prompt, "text"):
            return prompt.text
        return str(prompt)

    def generate_text(
        self,
        prompt: Any,
        n: int = 1,
        temperature: float = 1e-8,
        stop: Optional[List[str]] = None,
        callbacks: Optional[Any] = None,
    ) -> SimpleLLMResult:
        prompt_text = self._to_text(prompt)
        messages = [{"role": "user", "content": prompt_text}]

        # 한 프롬프트에 대한 n개 생성 → [ [g1, g2, g3] ] 구조
        gens_for_single_prompt: List[SimpleGeneration] = []

        for _ in range(max(n, 1)):
            resp = self.client.chat(
                model=self.model,
                messages=messages,
                options={
                    "temperature": max(temperature, 0.0),
                },
            )
            text = resp["message"]["content"]

            if stop:
                for s in stop:
                    idx = text.find(s)
                    if idx != -1:
                        text = text[:idx]
                        break

            gens_for_single_prompt.append(SimpleGeneration(text=text))

        # 바깥 리스트: 프롬프트 하나, 안쪽 리스트: 그 프롬프트에 대한 여러 생성
        return SimpleLLMResult(generations=[gens_for_single_prompt])

    async def agenerate_text(
        self,
        prompt: Any,
        n: int = 1,
        temperature: float = 1e-8,
        stop: Optional[List[str]] = None,
        callbacks: Optional[Any] = None,
    ) -> SimpleLLMResult:
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(
            None,
            self.generate_text,
            prompt,
            n,
            temperature,
            stop,
            callbacks,
        )

    def is_finished(self, response: SimpleLLMResult) -> bool:
        try:
            for gen_list in response.generations:
                for gen in gen_list:
                    if not getattr(gen, "text", "").strip():
                        return False
            return True
        except Exception:
            return False


# -------------------------------------------------------------------
# 3) SentenceTransformer 기반 로컬 임베딩 (BaseRagasEmbeddings 구현)
# -------------------------------------------------------------------
class LocalHFEmbeddings(BaseRagasEmbeddings):
    def __init__(self, model_name: str = "intfloat/multilingual-e5-large-instruct"):
        super().__init__()
        # ragas 내부 로깅에서 embeddings.model 을 문자열로 기대하므로 이렇게
        self.model = model_name
        # 실제 HF 모델은 별도 속성에
        self._model = SentenceTransformer(model_name)

    # 동기 버전
    def embed_query(self, text: str) -> List[float]:
        emb = self._model.encode([text], normalize_embeddings=True)[0]
        return emb.tolist()

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        if len(texts) == 0:
            return []
        embs = self._model.encode(texts, normalize_embeddings=True)
        return [e.tolist() for e in embs]

    # 비동기 버전
    async def aembed_query(self, text: str) -> List[float]:
        return self.embed_query(text)

    async def aembed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.embed_documents(texts)


# -------------------------------------------------------------------
# 4) 인스턴스 생성
# -------------------------------------------------------------------
# 평가용 LLM: 기본은 8B, 꼭 20B로 평가까지 하고 싶으면 model="gpt-oss:20b" 로 바꾸면 됨
llm = OllamaRagasLLM(
    model="gpt-oss:20b",
    host="http://127.0.0.1:11434",
)

embeddings = LocalHFEmbeddings(
    model_name="intfloat/multilingual-e5-large-instruct",
)

# -------------------------------------------------------------------
# 5) 메트릭 정의
# -------------------------------------------------------------------
metrics = [context_precision, context_recall, faithfulness, answer_relevancy]

# -------------------------------------------------------------------
# 6) run_config: 동시 작업 수 / 타임아웃 조절
# -------------------------------------------------------------------
run_config = RunConfig(
    timeout=1200,
    max_workers = 8  
)

# -------------------------------------------------------------------
# 7) 평가 실행
# -------------------------------------------------------------------
result = evaluate(
    dataset=hf_ds,
    metrics=metrics,
    llm=llm,
    embeddings=embeddings,
    run_config=run_config,
)

print(result)
result_df = result.to_pandas()
result_df.head()
# result_df.to_csv("ragas_eval_result.csv", index=False)

Evaluating: 100%|██████████| 120/120 [32:21<00:00, 16.18s/it]


{'context_precision': 0.6000, 'context_recall': 0.4833, 'faithfulness': 0.3833, 'answer_relevancy': 0.6452}


,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,"Who is the current CEO of Setopia Co., Ltd.?",[대표이사 (성 명) 서상철],"The current CEO of Setopia Co., Ltd. is 서상철.","The current CEO of Setopia Co., Ltd. is Sang-c...",1.0,1.0,0.000000,1.000000
1,"What was the company name of Setopia Co., Ltd....",[2022년 8월 1일 임시주주총회 이후 (주)마이더스AI에서 (주)세토피아로 상호...,It was (주)마이더스AI.,"It was Midas AI Co., Ltd.",1.0,1.0,1.000000,0.874220
2,"When was the merger date for Setopia Co., Ltd....",[],I cannot answer the question based on the prov...,"The merger date was January 2, 2023.",0.0,0.0,0.000000,0.000000
3,What is the main product of Setopia's steel bu...,"[철강사업 당사는 철강, 스테인레스강, 특수강 등의 도매, 제조사업을 영위하고 있으...",Setopia’s steel business mainly sells stainles...,"The main product is STS 201, and it holds the ...",0.0,0.0,0.333333,0.961088
4,The distribution business Setopia entered into...,[2월 글로벌 1위 전자담배브랜드 ELFBAR(엘프바) 제품 국내 총판 독점 계약을...,ELFBAR (엘프바),It was an exclusive domestic distributorship c...,1.0,1.0,1.000000,0.800566


In [23]:
result

{'context_precision': 0.6000, 'context_recall': 0.4833, 'faithfulness': 0.3833, 'answer_relevancy': 0.6452}

In [24]:
result_df

,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,"Who is the current CEO of Setopia Co., Ltd.?",[대표이사 (성 명) 서상철],"The current CEO of Setopia Co., Ltd. is 서상철.","The current CEO of Setopia Co., Ltd. is Sang-c...",1.0,1.0,0.000000,1.000000
1,"What was the company name of Setopia Co., Ltd....",[2022년 8월 1일 임시주주총회 이후 (주)마이더스AI에서 (주)세토피아로 상호...,It was (주)마이더스AI.,"It was Midas AI Co., Ltd.",1.0,1.0,1.000000,0.874220
2,"When was the merger date for Setopia Co., Ltd....",[],I cannot answer the question based on the prov...,"The merger date was January 2, 2023.",0.0,0.0,0.000000,0.000000
3,What is the main product of Setopia's steel bu...,"[철강사업 당사는 철강, 스테인레스강, 특수강 등의 도매, 제조사업을 영위하고 있으...",Setopia’s steel business mainly sells stainles...,"The main product is STS 201, and it holds the ...",0.0,0.0,0.333333,0.961088
4,The distribution business Setopia entered into...,[2월 글로벌 1위 전자담배브랜드 ELFBAR(엘프바) 제품 국내 총판 독점 계약을...,ELFBAR (엘프바),It was an exclusive domestic distributorship c...,1.0,1.0,1.000000,0.800566
5,Who was the auditor for the 11th fiscal period...,[삼일회계법인 의견거절],"The auditor was 삼일회계법인, and the audit opinion ...","The auditor was Samil PwC, and the audit opini...",1.0,1.0,0.000000,0.934677
6,What happened to the contract for the acquisit...,[해지 하는 것으로 결정하게 되었습니다 계약상대방 (주)에스에이코퍼레이션 2. 자산...,The contract for acquiring the land and buildi...,"The contract was terminated on January 20, 2025.",1.0,0.0,0.000000,0.938608
7,What was the ratio of the capital reduction wi...,"[13,332,272 88.11% 합 계 15,131,870 100.00%]",The ratio was 88.11%.,A 5-to-1 capital reduction was completed.,0.0,0.0,0.000000,0.862220
8,"Who is the largest shareholder of Setopia Co.,...","[(주)에스에이코퍼레이션 1,042,986 6.89%]","The largest shareholder is (주)에스에이코퍼레이션, holdi...","The largest shareholder is SA Corporation Co.,...",1.0,1.0,0.000000,0.948104
9,"How much was the fine imposed on Setopia Co., ...",[],I cannot answer the question based on the prov...,A fine of 270 million KRW was imposed related ...,0.0,0.0,0.000000,0.000000


In [25]:
result_df.to_csv('result.csv')